In [37]:
import numpy as np
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests
from supabase import create_client
from datetime import datetime

In [18]:

SUPABASE_URL = "https://lbiioyctwrwxwwbnaewa.supabase.co"
SUPABASE_KEY = "sb_publishable_4ASf43j7RR3rE2c-DumiIg_4Tc5UvqH"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)


In [20]:
response = (
    supabase
    .table("company_contacts")
    .select("*")
    .range(14700,14852  )
    .execute()
)

In [23]:
data = response.data
df = pd.DataFrame(data)

In [25]:

for row in response.data:
    supabase.table("company_contacts").update({"reference": "Real Estate Batch - 1"}).eq("id", row["id"]).execute()


In [32]:
df = pd.read_excel("whatsapp_upload.xlsx")

In [33]:
df

,company,state,CIN,ZAUBA CO,DIRECTOR NAME,DIN,addrress,Target,Phone Number,Email,Odoo Phone Number,Odoo Email
0,Baderwals Infraprojects Private Limited,Haryana,U70109DL2006PTC151001,BADERWALS INFRAPROJECTS PRIVATE LIMITED,Sushil Bhardwaj,1725433,211 ANSAL BHAWAN 16 K G MARG NEW DELHI DL 1100...,Y,9.198737e+11,sanjeevtushar0101@gmail.com,9873666381,sanjeevtushar0101@gmail.com
1,Saera Auto India Private Limited,Haryana,U29219DL2006PTC155955,SAERA AUTO INDIA PRIVATE LIMITED,Nitin Kapoor,2069524,"Plot No. 1, Sector-11 Dwarka New Delhi South W...",NaN,9.715018e+11,india.jbt@emaar.ae,9810001305,satra_nitin@yahoo.co.in
2,Orris Infrastructure Private Limited,Haryana,U70109DL2006PTC151295,ORRIS INFRASTRUCTURE PRIVATE LIMITED,Vijay Gupta,6558,RZ-D-5 Mahavir Enclave New Delhi South West De...,Y,9.198111e+11,CMD@ORRIS.IN,9811065727,cmd@orris.in
3,ATS Estates Private Limited,"Andhra Pradesh, Punjab",U70102DL2007PTC160904,ATS ESTATES PRIVATE LIMITED,Naveen Joshi,7922648,"71192, DEEPALI NEHRU PLACE NEW DELHI DL 110019 IN",NaN,NaN,NaN,9990079075,naveenjoshiats@gmail.com
4,Imperia Structures Limited,"Haryana, Uttar Pradesh",U45400DL2010PLC198791,IMPERIA STRUCTURES LIMITED,Brajinder Singh Batra,47184,"A - 25, MOHAN CO - OPERATIVE INDUSTRIAL ESTATE...",Y,NaN,NaN,9810056500,brajinder@imperiastructures.com
...,...,...,...,...,...,...,...,...,...,...,...,...
153,G K Enterprise,Delhi,U74899DL1991PTC045971,GK ENTERPRISES PRIVATE LIMITED,HITESH AGARWAL,1058239,3443 FIRST FLOORDARIBA PAN PAHARGANJ NEW DELHI...,NaN,NaN,NaN,9810353646,NaN
154,Vaastu Vaibhav Constructions,Delhi,U45201DL2005PTC131908,VASTU VAIBHAV BUILDCON PRIVATE LIMITED,VIJAY KUMAR SINGHAL,1736879,"48, Second Floor East Azad Nagar Delhi East De...",NaN,NaN,NaN,9910410004,NaN
155,Kavshallays Industries Sahkari Sanstha,Delhi,U29100HR2020PTC087226,KOUSHALYA INDUSTRIES PRIVATE LIMITED,MAHESH AGARWAL,3115137,"Plot No. 68, Gali No. 2E, Sarurpur Industrial...",NaN,NaN,NaN,9344611129,NaN
156,THE Swaraj Realtors,Delhi,U70102DL2006PTC156529,SWARAJ REALTORS PRIVATE LIMITED,NILESH KUMAR SINGH,54983,"29A, POCKET 1, SECTOR 7 DWARKA NEW DELHI DL 11...",NaN,NaN,NaN,9311303636,NaN


In [ ]:

def clean_value(value):
    if pd.isna(value) or str(value).strip() == "":
        return None
    return value

In [26]:

headers = {
  "token": "webxjdlznb",
  "Content-Type": "application/json"
}

In [35]:

for i in range(len(df)):

    contact_response = (
        supabase
        .table("company_contacts")
        .select("id")
        .eq("name", df.loc[i, "DIRECTOR NAME"])
        .limit(1)
        .execute()
    )

    contact_id = contact_response.data[0]["id"] if contact_response.data else None
    print(contact_id)
    phone_number = df.loc[i, "Odoo Phone Number"][-10:]

    payload = {
        "sender_whatsapp_number": phone_number,
        "template_name": "labour_complianc",
        "broadcast_name": "Labour_complaince_batch1",
        "url": "https://smartchatapi.live/portal/uploads/header_media/1788240440.jpg",
        "parameter_value1": ""
    }

    response = requests.post(
        "https://smartchatapi.live/portal/Api/send_template_message",
        headers=headers,
        json=payload
    )

    # If 400, add 91 and retry
    if response.status_code == 400:

        phone_number = "91" + phone_number

        payload["sender_whatsapp_number"] = phone_number

        print(f"400 received. Retrying with: {phone_number}")

        response = requests.post(
            "https://smartchatapi.live/portal/Api/send_template_message",
            headers=headers,
            json=payload
        )

    print("Status Code:", response.status_code)

    try:
        data1 = response.json()
        print("Response:")
        print(data1)

    except Exception:
        data1 = None
        print("Raw Response:")
        print(response.text)
        print(df.loc[i, "DIRECTOR NAME"])

    if response.status_code == 200:

        print("Message Sent!")

        data1 = {
            "company_id": 6,
            "contact_id": contact_id,
            "name": df.loc[i, "DIRECTOR NAME"],
            "phone_number": phone_number,
            "company_name": df.loc[i, "ZAUBA CO"],
            "template_name": "labour_complianc",
            "sent_at": datetime.now().isoformat(),
            "Broadcast_name": "Labour_complaince_batch1"
        }

        supabase.table(
            "whatsapp_message_logs"
        ).insert(data1).execute()

    else:
        st.error("Failed")

    print(df.loc[i, "DIRECTOR NAME"])
    print(i)


15737


IndexError: invalid index to scalar variable.

In [40]:
contact_response

APIResponse(data=[{'id': 3}], count=None)

In [ ]:

for i in range(1):

    contact_response = (
        supabase
        .table("company_contacts")
        .select("id, company_id")
        .eq("name", "Simran")
        .limit(1)
        .execute()
    )

    contact_id = contact_response.data[0]["id"] if contact_response.data else None
    company_id = contact_response.data[0]['company_id'] if contact_response.data else None
    phone_number = "7982166763"[-10:]

    payload = {
        "sender_whatsapp_number": phone_number,
        "template_name": "labour_complianc",
        "broadcast_name": "Labour_complaince_batch1",
        "url": "https://smartchatapi.live/portal/uploads/header_media/1788240440.jpg",
        "parameter_value1": ""
    }

    response = requests.post(
        "https://smartchatapi.live/portal/Api/send_template_message",
        headers=headers,
        json=payload
    )

    # If 400, add 91 and retry
    if response.status_code == 400:

        phone_number = "91" + phone_number

        payload["sender_whatsapp_number"] = phone_number

        print(f"400 received. Retrying with: {phone_number}")

        response = requests.post(
            "https://smartchatapi.live/portal/Api/send_template_message",
            headers=headers,
            json=payload
        )

    print("Status Code:", response.status_code)

    try:
        data1 = response.json()
        print("Response:")
        print(data1)

    except Exception:
        data1 = None
        print("Raw Response:")
        print(response.text)
        print(df.loc[i, "DIRECTOR NAME"])

    if response.status_code == 200:

        print("Message Sent!")

        data1 = {
            "company_id": company_id,
            "contact_id": contact_id,
            "name":"Simran",
            "phone_number": phone_number,
            "company_name": df.loc[i, "ZAUBA CO"],
            "template_name": "labour_complianc",
            "sent_at": datetime.now().isoformat(),
            "Broadcast_name": "Labour_complaince_batch1"
        }

        supabase.table(
            "whatsapp_message_logs"
        ).insert(data1).execute()

    else:
        st.error("Failed")

    print(df.loc[i, "DIRECTOR NAME"])
    print(i)


KeyError: 'company_id'